<a href="https://colab.research.google.com/github/wav-ent/gdg-brisbane-demo-may2026/blob/main/Queensland_AI_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# LLM Quality Eval Pipeline
Compares Human vs AI-Generated Python code using Claude as a judge

In [1]:

!pip install pandas
!pip install anthropic
!pip install IPython
!pip install scipy
!pip install pathlib

In [2]:
# %% Import Functions
import os
import re
import time
import pandas as pd
import anthropic
from dotenv import load_dotenv
from IPython.display import display, Markdown
from scipy import stats
from pathlib import Path
from google.colab import userdata


In [3]:
from google.colab import files
uploaded = files.upload()  # opens a file picker

Saving QLD_AI_Demo_Pipeline.csv to QLD_AI_Demo_Pipeline.csv


In [4]:
# %% Configuration
try:
    WORKING_DIR = Path(__file__).parent
except NameError:  # running interactively
    WORKING_DIR = Path.cwd()

FILE_NAME   = "QLD_AI_Demo_Pipeline.csv"
MODEL       = "claude-sonnet-4-5"

# %% Read in Data
exampleData = pd.read_csv(
    os.path.join(WORKING_DIR, FILE_NAME),
    encoding='utf-8',
    skipinitialspace=True
).dropna(how='all')


print(f"Loaded {len(exampleData)} rows")
display(exampleData[['Example', 'Task Description', 'Human_Written_Code']]
        .style
        .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})
        .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}])
        .hide(axis='index')
)

# %% Initialize Anthropic Client
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

Loaded 5 rows


Example,Task Description,Human_Written_Code
1,We need to build a function in python that we can use to print a statement 5 times,"def print_statement(text: str) -> None: print(f""{text}!\n"" * 5)"
2,"Adjust an existing function read_and_prep_data so that we first read in dates as a string, and then convert to datetime","def read_and_prep_data(file_path: str) -> pd.DataFrame: df = pd.read_csv(file_path, dtype={""date"": str}) df[""date""] = pd.to_datetime(df[""date""]) return df"
3,"Read in an empty file_path """" and then filter out the column ""Price"" for Nulls","file_path = """" df = pd.read_csv(file_path) df = df.dropna(subset=[""Price""])"
4,"Read in an empty file_path """" and then filter out the column ""Price"" for Nulls Aggregate Column Price and Column Tax into a new column for the dataframe","file_path = """" df = pd.read_csv(file_path) df = df.dropna(subset=[""Price""]) df[""Total_Price""] = df[""Price""] + df[""Tax""]"
5,Combine Price Table with Cost Data based on the keys 'SKU',"price_file_path = """" price_df = pd.read_csv(price_file_path) price_df = price_df.dropna(subset=[""Price""]) price_df[""Total_Price""] = price_df[""Price""] + price_df[""Tax""] cost_file_path = """" cost_df = pd.read_csv(cost_file_path) combined_df = pd.merge(price_df, cost_df, on=""SKU"", how=""left"")"


## PART 1: Define a Function to Score Two Code Snippers (Human vs AI) using Claude


In [5]:
def get_llm_score(row, ai_code_col, objective_code_col = 'Human_Written_Code') :
    human_code = row[objective_code_col]
    ai_code    = row[ai_code_col]
    task_desc  = row['Task Description']

    prompt = f"""
    Role: You are a Senior Python Architect and Static Analysis Expert.
    Your task is to perform a structural audit comparing [Human Code] (the gold
    standard for efficiency) and [AI Code].

    Task Description: {task_desc}

    [Human Code]
    {human_code}

    [AI Code]
    {ai_code}

    Objective: Provide a single Similarity Score (0-100%) based on how closely
    the AI's underlying logic and structural efficiency mirror the human's code.

    Evaluation Criteria:
    - Structural Alignment: Do the two versions use the same control flow?
    - Algorithmic Parity: Does the AI use the same optimised logic? Penalise
      heavily for inferior data structures (e.g. list vs set for lookups).
    - Complexity Matching: Penalise unnecessary complexity or code smell.

    Ignore: variable/function names, imports, docstrings, comments, whitespace.

    Output Requirement:
    Respond with ONLY this exact format and nothing else:
    Similarity Score: [X]%
    """

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=256,
            temperature=0,
            messages=[{"role": "user", "content": prompt}]
        )
        output_text = response.content[0].text
       # print(f"\n  DEBUG response: {output_text!r}")  # temporary debug line
        match = re.search(r'Similarity Score:\s*(\d+)\s*%', output_text)
        return f"{match.group(1)}%" if match else "Format Error"

    except Exception as e:
        print(f"  Error scoring row: {e}")
        return "API Error"

## PART 2: Run AI Code Generation (Basic and  Advanced Prompt)

In [6]:
def generate_ai_code(row, prompt_type="advanced"):
    """Generate Python code from a task description using Claude."""
    task_desc = row['Task Description']

    prompts = {
        "basic": f"""
        You are learning Python. Your code usually has mistakes. It is not efficient.
        Write a simple Python solution for the following task:
        Task: {task_desc}
        Rules:
        - Return ONLY raw Python code
        - No markdown, no code fences, no comments, no docstrings
        """,
        "advanced": f"""
        You are an expert Python developer.
        Write a clean, efficient Python solution for the following task:
        Task: {task_desc}
        Rules:
        - Return ONLY raw Python code
        - No markdown, no code fences, no comments, no docstrings
        """
    }

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=512,
            temperature=0,
            messages=[{"role": "user", "content": prompts[prompt_type]}]
        )
        raw = response.content[0].text.strip()
        # Strip markdown code fences if present
        raw = re.sub(r'^```[\w]*\n?', '', raw)  # Remove opening ```python or ```
        raw = re.sub(r'\n?```$', '', raw)       # Remove closing ```
        return raw.strip()

    except Exception as e:
        print(f"  Error generating code: {e}")
        return "API Error"

## PART 3: Generate Code using Both Basic and Advanced Prompts

In [7]:
print("\n" + "="*60)
print("PART 3: Generating API Code (Basic & Advanced)")
print("="*60)

for index, row in exampleData.iterrows():
    print(f"  Generating code for row {index}...")
    exampleData.at[index, 'AI_Code_Basic']    = generate_ai_code(row, prompt_type="basic")
    exampleData.at[index, 'AI_Code_Advanced'] = generate_ai_code(row, prompt_type="advanced")
    time.sleep(1)

print("Code generation complete!")


PART 3: Generating API Code (Basic & Advanced)
  Generating code for row 0...
  Generating code for row 1...
  Generating code for row 2...
  Generating code for row 3...
  Generating code for row 4...
Code generation complete!


## PART 4: Score API-Generated Code vs Human

In [8]:
print("\n" + "="*60)
print("PART 4: Scoring API Code vs Human")
print("="*60)

for index, row in exampleData.iterrows():
    print(f"  Scoring row {index}...")
    exampleData.at[index, 'Score_Basic']    = get_llm_score(row, ai_code_col='AI_Code_Basic')
    exampleData.at[index, 'Score_Advanced'] = get_llm_score(row, ai_code_col='AI_Code_Advanced')
    time.sleep(1)


PART 4: Scoring API Code vs Human
  Scoring row 0...
  Scoring row 1...
  Scoring row 2...
  Scoring row 3...
  Scoring row 4...


## PART 5: Final Results

In [9]:
pd.set_option('display.max_colwidth', None)

# %% Compute Scores
basic_scores    = exampleData['Score_Basic'].str.replace('%', '').astype(float)
advanced_scores = exampleData['Score_Advanced'].str.replace('%', '').astype(float)

baseline_quality_score = basic_scores.mean()
advanced_quality_score = advanced_scores.mean()

# %% Summary Header
display(Markdown("# Part 5: Final Results"))
display(Markdown(f"""
### Summary
| Metric | Score |
|--------|-------|
| Average Basic Prompt Score | {baseline_quality_score:.1f}% |
| Average Advanced Prompt Score | {advanced_quality_score:.1f}% |
"""))

# %% Per-row Results
for index, row in exampleData.iterrows():
    display(Markdown(f"---\n## Row {index}: {row['Task Description']}"))
    display(Markdown(f"**Human Written Code:**\n```python\n{row['Human_Written_Code']}\n```"))
    display(Markdown(f"**Basic** — Score: `{row['Score_Basic']}`\n```python\n{row['AI_Code_Basic']}\n```"))
    display(Markdown(f"**Advanced** — Score: `{row['Score_Advanced']}`\n```python\n{row['AI_Code_Advanced']}\n```"))

# %% Write to Disk
display(Markdown("---\n# Part 6: Write Results to Disk"))
exampleData.to_csv(
    os.path.join(WORKING_DIR, "QLD_AI_Demo_Pipeline.csv"),
    index=False,
    encoding='utf-8'
)
display(Markdown("✅ Results saved to `QLD_AI_Demo_Pipeline.csv`"))

# %% Statistical Significance
t_stat, p_value = stats.ttest_rel(basic_scores, advanced_scores)
sig = "✅ Yes" if p_value < 0.05 else "❌ No"

display(Markdown(f"""
---
### Statistical Significance (Paired T-Test)
| | Value |
|--|--|
| T-statistic | `{t_stat:.4f}` |
| P-value | `{p_value:.4f}` |
| Significant at 95% confidence? | {sig} |
"""))

# Part 5: Final Results


### Summary
| Metric | Score |
|--------|-------|
| Average Basic Prompt Score | 64.4% |
| Average Advanced Prompt Score | 70.4% |


---
## Row 0: We need to build a function in python that we can use to print a statement 5 times

**Human Written Code:**
```python
def print_statement(text: str) -> None:
    print(f"{text}!\n" * 5)
```

**Basic** — Score: `25%`
```python
def print_statement():
    for i in range(5):
        print("This is a statement")

print_statement()
```

...

**Basic** — Score: `15%`
```python
price_table = [
    {'SKU': 'A001', 'Price': 10.5},
    {'SKU': 'A002', 'Price': 20.0},
    {'SKU': 'A003', 'Price': 15.75}
]

cost_data = [
    {'SKU': 'A001', 'Cost': 5.0},
    {'SKU': 'A002', 'Cost': 12.0},
    {'SKU': 'A004', 'Cost': 8.0}
]

result = []
for price_item in price_table:
    for cost_item in cost_data:
        if price_item['SKU'] == cost_item['SKU']:
            combined = {}
            combined['SKU'] = price_item['SKU']
            combined['Price'] = price_item['Price']
            combined['Cost'] = cost_item['Cost']
            result.append(combined)

print(result)
```

**Advanced** — Score: `45%`
```python
import pandas as pd

def combine_data(price_table, cost_data):
    return pd.merge(price_table, cost_data, on='SKU', how='outer')
```

---
# Part 6: Write Results to Disk

✅ Results saved to `QLD_AI_Demo_Pipeline.csv`


---
### Statistical Significance (Paired T-Test)
| | Value |
|--|--|
| T-statistic | `-1.0000` |
| P-value | `0.3739` |
| Significant at 95% confidence? | ❌ No |
